In [2]:
import requests
import pandas as pd

'''
@oid
@sign
@list
@ucun
@v
@link	eBL
'''

file_url = "https://raw.githubusercontent.com/oracc/osl/master/00lib/osl.asl"

response = requests.get(file_url)
response.raise_for_status()
file_content = response.text

rows = []
current = {}
current_ucun = None
variant_index = -1
emitted_since_ucun = False

for line in file_content.splitlines():
    if line.startswith("@oid") and "\t" in line:
        oid = line.split("\t", 1)[1].strip()
        n = 1
        while f"oid_{n}" in current:
            n += 1
        current[f"oid_{n}"] = oid

    elif line.startswith("@sign"):
        current = {"sign": line.split()[1].strip()}
        current_ucun = None
        variant_index = -1
        emitted_since_ucun = False

    elif line.startswith("@list ") and "\t" in line:
        current["list"] = line.split("\t", 1)[1].strip()

    elif line.startswith("@ucun") and "\t" in line:
        if current_ucun is not None and not emitted_since_ucun:
            base = {k: v for k, v in current.items() if not isinstance(v, (list, dict))}
            base.update({"ucun": current_ucun, "value": None, "variant_index": variant_index})
            rows.append(base)

        current_ucun = line.split("\t", 1)[1].strip()
        variant_index += 1
        emitted_since_ucun = False

    elif line.startswith("@v") and "\t" in line:
        val = line.split("\t", 1)[1].strip()
        base = {k: v for k, v in current.items() if not isinstance(v, (list, dict))}
        base.update({"ucun": current_ucun, "value": val, "variant_index": variant_index})
        rows.append(base)
        emitted_since_ucun = True

    elif line.startswith("@link\teBL") and "\t" in line:
        current["link_eBL"] = line.split("\t", 1)[1].strip()

    elif line.startswith("@end sign"):
        if current_ucun is not None and not emitted_since_ucun:
            base = {k: v for k, v in current.items() if not isinstance(v, (list, dict))}
            base.update({"ucun": current_ucun, "value": None, "variant_index": variant_index})
            rows.append(base)
        current = {}
        current_ucun = None
        variant_index = -1
        emitted_since_ucun = False

Osl_df = (
    pd.DataFrame(rows)
      .rename(columns={
          'sign':  'OSL(form)',
          'ucun':  'OSL(sign)',
          'value': 'OSL(value)'
      })
)

preferred_cols = [
    "oid_1","oid_2","oid_3",
    "OSL(form)","list",
    "OSL(sign)","OSL(value)",
    "link_eBL","variant_index"
]
Osl_df = Osl_df[[c for c in preferred_cols if c in Osl_df.columns]]

Osl_df


,oid_1,oid_2,oid_3,OSL(form),OSL(sign),OSL(value),link_eBL,variant_index
0,o0000087,NaN,NaN,A,𒀀,ʾu₄,NaN,0
1,o0000087,NaN,NaN,A,𒀀,a,NaN,0
2,o0000087,NaN,NaN,A,𒀀,aia₂,NaN,0
3,o0000087,NaN,NaN,A,𒀀,aya₂,NaN,0
4,o0000087,NaN,NaN,A,𒀀,barₓ,NaN,0
...,...,...,...,...,...,...,...,...
12274,o0038262,NaN,NaN,7(|GEŠU@c×KASKAL|),󰀱,7(ŋešʾu@v),NaN,0
12275,o0027234,NaN,NaN,NaN,None,1(aš@f),NaN,-1
12276,o0027234,o0027294,o0027327,NaN,None,2(u@f),NaN,-1
12277,o0027234,o0027294,o0027327,NaN,None,3(u@f),NaN,-1
